# Swarm Clock, Part 2b: lineage lifespans, exploratory temporal test, and calibration
Needs `data/wiki_intros.parquet` from Part 2a (same runtime, or re-upload it).
Primary configuration (frozen): k=3 token shingles, LSH threshold 0.3, hubs excluded. The other configurations are sensitivity analyses.
Temporal statistic (root-free, date-blind families): within each family, Spearman correlation between pairwise text distance (1 - Jaccard) and pairwise |time difference|; averaged over families; null = permute times among block representatives within date-blind strata (family x length tercile). One-sided.

In [ ]:
!pip -q install datasketch
import re, collections, numpy as np, pandas as pd, matplotlib.pyplot as plt
from datasketch import MinHash, MinHashLSH
from scipy.stats import rankdata
rng = np.random.default_rng(20261108)
F = pd.read_parquet("data/wiki_intros.parquet")
HUBS = {"dse/WillkommenImWiki","dse/StartSeite","dse/TestSeite","dse/RecentChanges"}
R = F[~F.page_id.isin(HUBS)].reset_index(drop=True)[["rev_id","page_id","label","t","m"]]
R["t"] = pd.to_datetime(R.t, utc=True)
print("real introductions (hubs excluded):", len(R))
NPERM = 128

In [ ]:
def shingle_set(t, k):
    toks = t.split()
    return {" ".join(toks[j:j+k]) for j in range(max(1, len(toks)-k+1))}
def minhash(s):
    m = MinHash(num_perm=NPERM)
    for x in s: m.update(x.encode())
    return m
CACHE = {}
def real_minhashes(k):
    if k not in CACHE: CACHE[k] = [minhash(shingle_set(t, k)) for t in R.m]
    return CACHE[k]
def assign_families(mhs, thr):
    lsh = MinHashLSH(threshold=thr, num_perm=NPERM)
    for i, m in enumerate(mhs): lsh.insert(str(i), m)
    par = list(range(len(mhs)))
    def fd(x):
        while par[x] != x: par[x] = par[par[x]]; x = par[x]
        return x
    for i, m in enumerate(mhs):
        for j in lsh.query(m):
            a, b = fd(i), fd(int(j))
            if a != b: par[a] = b
    return np.array([fd(i) for i in range(len(mhs))])
def family_table(df, fam):
    D = df.assign(family=fam)
    g = D.groupby("family").agg(blocks=("page_id","nunique"), labels=("label","nunique"), t0=("t","min"), t1=("t","max"))
    g["span_h"] = (g.t1 - g.t0).dt.total_seconds()/3600
    g["days"] = D.groupby("family").t.apply(lambda s: s.dt.date.nunique())
    return D, g

## 1. Lineage lifespans across similarity definitions (Finding B)

In [ ]:
CONFIGS = [("k5 t0.5",5,.5), ("k5 t0.3",5,.3), ("k3 t0.3 (primary)",3,.3), ("k3 t0.2",3,.2)]
spans, summary = {}, []
for name, k, thr in CONFIGS:
    fam = assign_families(real_minhashes(k), thr)
    D, g = family_table(R, fam)
    f5 = g[g.blocks >= 5]
    spans[name] = f5.span_h.values
    summary.append({"config": name, "families>=5 blocks": len(f5), "eligible(>=15)": int((g.blocks>=15).sum()),
                    "multi_day(>=3d, >=15 blocks)": int(((g.blocks>=15)&(g.days>=3)).sum()),
                    "median_span_min": round(60*np.median(f5.span_h),1), "p75_h": round(np.quantile(f5.span_h,.75),2),
                    "p90_h": round(np.quantile(f5.span_h,.9),2), "share_>24h": round((f5.span_h>24).mean(),3),
                    "largest_family_share": round(g.blocks.max()/g.blocks.sum(),3)})
    if "primary" in name: D_primary, G_primary = D, g
S = pd.DataFrame(summary); print(S.to_string(index=False)); S.to_csv("data/finding_B_lifespans.csv", index=False)
plt.figure(figsize=(6,4))
for name, v in spans.items():
    x = np.sort(np.clip(v, 1/60, None)); plt.step(x, np.arange(1, len(x)+1)/len(x), where="post", label=name)
plt.xscale("log"); plt.xlabel("lineage lifespan (hours, log)"); plt.ylabel("ECDF (families with >= 5 pages)")
for xv, lab in [(1/6,"10 min"),(1,"1 h"),(24,"1 day")]: plt.axvline(xv, ls=":", c="gray"); plt.text(xv, .02, lab, fontsize=8)
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig("fig_lifespans.png", dpi=200); plt.show()

## 2. Characterize the eligible families in the primary configuration

In [ ]:
elig = G_primary[G_primary.blocks >= 15].sort_values("blocks", ascending=False)
print(elig[["blocks","labels","t0","t1","span_h","days"]].to_string())
for f in elig.index:
    sub = D_primary[D_primary.family == f].sort_values("t")
    print(f"\n=== family {f}: {len(sub)} intros, {sub.page_id.nunique()} pages, {sub.label.nunique()} labels")
    for q in [0, len(sub)//2, len(sub)-1]:
        print("  ", sub.t.iloc[q], "|", sub.m.iloc[q][:200])

## 3. Temporal association test (exploratory on real data; see calibration below)

In [ ]:
def reps_for(D, fam_ids):
    X = D[D.family.isin(fam_ids)].copy()
    X["len"] = X.m.str.len()
    X = X.sort_values(["family","page_id","len","rev_id"], ascending=[True,True,False,True]).drop_duplicates(["family","page_id"])
    X["stratum"] = X.groupby("family")["len"].transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=False) if len(s) >= 6 else 0)
    return X
def family_arrays(X, k=3):
    out = []
    for f, sub in X.groupby("family"):
        sets = [shingle_set(t, k) for t in sub.m]; n = len(sets)
        iu = np.triu_indices(n, 1)
        d = np.array([1 - len(sets[i] & sets[j]) / max(1, len(sets[i] | sets[j])) for i, j in zip(*iu)])
        th = sub.t.astype("int64").values / 3.6e12
        out.append({"f": f, "rd": rankdata(d), "t": th, "strata": sub.stratum.values, "iu": iu})
    return out
def stat(arrs, times=None):
    vals = []
    for a, t in zip(arrs, times or [a["t"] for a in arrs]):
        dt_ = np.abs(t[a["iu"][0]] - t[a["iu"][1]])
        if np.ptp(dt_) == 0 or np.ptp(a["rd"]) == 0: continue
        vals.append(np.corrcoef(a["rd"], rankdata(dt_))[0,1])
    return np.mean(vals) if vals else np.nan, vals
def perm_test(arrs, B=999):
    obs, per_family = stat(arrs)
    if np.isnan(obs): return np.nan, np.nan, per_family
    null = np.empty(B)
    for b in range(B):
        ts = []
        for a in arrs:
            t = a["t"].copy()
            for s in np.unique(a["strata"]):
                idx = np.where(a["strata"] == s)[0]; t[idx] = t[rng.permutation(idx)]
            ts.append(t)
        null[b] = stat(arrs, ts)[0]
    null = null[~np.isnan(null)]
    return obs, (1 + np.sum(null >= obs)) / (len(null) + 1), per_family
X_real = reps_for(D_primary, elig.index)
arrs_real = family_arrays(X_real)
T_obs, p_real, per_fam = perm_test(arrs_real, B=999)
print(f"REAL (exploratory): families={len(arrs_real)}  T={T_obs:.3f}  p={p_real:.3f}")
print("per-family rho:", np.round(per_fam, 3))

## 4. Calibration: would the pipeline find a multi-day clock if one existed?
Plant synthetic lineages into the real log, rerun the frozen primary pipeline, and measure:
- recovery: lineage lands in one family (>= 80% of members) with >= 15 pages over >= 3 days;
- power: temporal test significant (p < 0.05) on the recovered family when the lineage accumulates edits (`clock`);
- false positives: same test on lineages with no accumulation (`star`: every member is an independent edit of the root, times random).

In [ ]:
vocab = np.array([w for t in R.m.sample(min(3000, len(R)), random_state=0) for w in t.split()])
roots_pool = R[R.m.str.split().str.len().between(25, 120)].m.tolist()
T_LO, T_HI = R.t.min().value, R.t.max().value
def mutate(toks, p):
    out = []
    for w in toks:
        u = rng.random()
        if u < p/2: continue                              # deletion
        out.append(vocab[rng.integers(len(vocab))] if u < p else w)   # substitution
        if rng.random() < p/2: out.append(vocab[rng.integers(len(vocab))])  # insertion
    return out
def plant(kind, p, n_lin=20, L=30, days=5):
    rows = []
    for li in range(n_lin):
        root = roots_pool[rng.integers(len(roots_pool))].split()
        t0 = rng.integers(T_LO, T_HI - int(days*86400e9))
        times = np.sort(t0 + rng.integers(0, int(days*86400e9), L))
        members = []
        for i in range(L):
            if kind == "clock":
                parent = root if i == 0 else members[rng.integers(max(0, i-5), i)]   # copy from recent members
                members.append(mutate(parent, p))
            else:
                members.append(mutate(root, 3*p))
        if kind == "star": times = rng.permutation(times)
        for i in range(L):
            rows.append({"rev_id": f"SYN{kind}{p}_{li}_{i}", "page_id": f"SYN/{kind}{p}_{li}_{i}", "label": f"syn{li}_{i}",
                         "t": pd.Timestamp(int(times[i]), tz="UTC"), "m": " ".join(members[i]), "lin": li})
    return pd.DataFrame(rows)
def run_condition(kind, p, B=199):
    P = plant(kind, p)
    A = pd.concat([R.assign(lin=-1), P], ignore_index=True)
    mhs = real_minhashes(3) + [minhash(shingle_set(t, 3)) for t in P.m]
    fam = assign_families(mhs, 0.3)
    D, g = family_table(A[["rev_id","page_id","label","t","m"]], fam)
    D["lin"] = A.lin.values
    res = []
    for li in range(P.lin.max()+1):
        mem = D[D.lin == li]; f = mem.family.mode().iloc[0]
        complete = (mem.family == f).mean()
        recovered = complete >= .8 and g.loc[f,"blocks"] >= 15 and g.loc[f,"days"] >= 3
        pval = np.nan
        if recovered:
            arr = family_arrays(reps_for(D, [f]))
            pval = perm_test(arr, B=B)[1]
        res.append({"kind": kind, "p": p, "lin": li, "completeness": complete, "recovered": recovered, "p_temporal": pval})
    return pd.DataFrame(res)
CAL = pd.concat([run_condition(kind, p) for kind in ["clock","star"] for p in [0.02, 0.05, 0.10, 0.20]], ignore_index=True)
summary_cal = CAL.groupby(["kind","p"]).agg(recovery=("recovered","mean"), completeness=("completeness","median"),
                                             sig_rate=("p_temporal", lambda s: np.mean(s.dropna() < .05) if s.notna().any() else np.nan))
print(summary_cal.round(2).to_string()); CAL.to_csv("data/calibration.csv", index=False)

## How to read this
- If `clock` lineages are recovered and significant at realistic edit rates, and `star` rarely triggers, then the method has power and a valid null, so the absence of multi-day lineages in the real data is evidence of absence (Finding B), not a method failure.
- The real-data temporal test (section 3) stays exploratory: it covers only the few eligible families.